## U1 - Drive and Camera Test Utility
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 20th, 2026\
References: this notebook is based on prior work by Derek Runberg from Sparkfun.

### About: This utility is used to test the setup of the AI Explorer's motor commands and camera.

## Camera setup involves a common sequence of python commands.
Step 1. Import libraries designed specifically for the camera, camera data, and visualizing the camera data (pictures).  
Step 2. Create a camera object that can be called to capture images.  
Step 3. Create jupyter laboratory widgets to visualize the camera data.  
Step 4. Link your camera object and the widget so that new images are automatically streamed.  
Step 5. Display the widgets. 

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 1. This cell imports all the libraries we'll use for this notebook

# Importing custom helpers for NMAIA robotics tasks on Jupyter Laboratory
from jupyter_utils import register_click_handler, register_dlink

# Importing graphical user interface libraries
import traitlets
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
from IPython.display import display


# Importing our custom library for our TraitletCamera
from jetcam_lite import TraitletCamera, bgr8_to_jpeg

In [ ]:
### Camera Setup ###
# Step 2. Instantiate and start the camera
camera = TraitletCamera()
camera.start()

# Step 3. Create the widget to display the image feed
image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)

# Step 4. Link the widget and the camera feed
# register_dlink (instead of calling traitlets.dlink() directly) keeps this cell
# safe to re-run: it won't stack a second live link onto the widget.
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# Step 5. Display the image widget
display(image_widget)

### We use following sequence to command our robot:
 Step 1. Import libraries designed specifically for your robot.   
 Step 2. Create the 'rvr' object from the sphero library, we can use this objcect as our interface to the robot.  
 Step 3. Issue commands to the rover's motors to move, this can be done multiple times.  

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 1. Import libraries designed specifically for the robot

# Import the Sphero RVR Tank Robot Software Development Kit 
from sphero_sdk import RawMotorModesEnum
from sphero_sdk import DriveFlagsBitmask
import time

# Import our shared robot connection helpers - ensures port isn't blocked
from robot_utils import get_rvr, close_if_exists

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 2. Create the 'rvr' object using our shared robot_utils helper.

# get_rvr() connects to the robot and wakes it up. If you re-run this cell,
# or if a connection already exists, it safely reuses it instead of opening
# a second one.
rvr = get_rvr()

Notice that `get_rvr()` takes care of waking up the robot for you automatically. Without it, your robot would fall asleep and not respond to your commands.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# STEP 3. Issue commands to the robot, this can be done multiple times. 

######## WILL CAUSE ROBOT MOTION: ENSURE ROBOT IS ON THE GROUND #########

# When you execute this cell, the robot will be commanded to move forward by the computer.
##### IMPORTANT: the RVR has a 2 second command timeout, that means that after 2 seconds the RVR will come to a stop if it has not received a new command. #####
rvr.raw_motors(
    left_mode=RawMotorModesEnum.forward.value,     # Set the left motor mode: "forward"
    left_duty_cycle=50,  # Valid duty cycle range is 0-255
    right_mode=RawMotorModesEnum.forward.value,    # Set the right motor mode: "forward
    right_duty_cycle=50  # Valid duty cycle range is 0-255
    )

In [ ]:
# Prepare the interface for the robot and wake up the robot so I'll be ready to receive commands
import traitlets
import ipywidgets as widgets

# create button widgets for each direction
button_layout = widgets.Layout(width='100px', height='80px', align_self='center')
stop_button = widgets.Button(description='stop', button_style='danger', layout=button_layout)
forward_button = widgets.Button(description='forward', layout=button_layout)
backward_button = widgets.Button(description='backward', layout=button_layout)
left_button = widgets.Button(description='left', layout=button_layout)
right_button = widgets.Button(description='right', layout=button_layout)

# display buttons
middle_box = widgets.HBox([left_button,image_widget, right_button], layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([forward_button, middle_box, backward_button,stop_button])
display(controls_box)

#define button event handler functions



def stop(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )
        
    
def step_forward(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.5)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def step_backward(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.reverse.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.reverse.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def step_left(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.reverse.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.forward.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
        )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

def step_right(change):
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.forward.value,
        left_duty_cycle=128,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.reverse.value,
        right_duty_cycle=128  # Valid duty cycle range is 0-255
    )
    time.sleep(.25)
    rvr.raw_motors(
        left_mode=RawMotorModesEnum.off.value,
        left_duty_cycle=0,  # Valid duty cycle range is 0-255
        right_mode=RawMotorModesEnum.off.value,
        right_duty_cycle=0  # Valid duty cycle range is 0-255
        )

#attach the event handler functions to specific button events
forward_button.on_click(step_forward)
backward_button.on_click(step_backward)
left_button.on_click(step_left)
right_button.on_click(step_right)
stop_button.on_click(stop)

## Section 3: Teleoperation

This section tests gamepad connectivity: left/right stick positions, the left/right bumpers, and driving the RVR from the gamepad via a single control-loop thread (see `gamepad_utils.py` for why a dedicated thread is used instead of acting directly on the gamepad library's own background thread).

The GUI below shows live stick position (sliders) and live bumper state (lit while held, dark when released) -- useful for confirming the controller is mapped correctly before relying on it in a student notebook. Bumpers are not wired to any capture/image logic here -- this section is connectivity/state testing only.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 1. Import our shared gamepad + teleop control loop helpers
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 2. Connect the gamepad, and build a GUI showing live stick position and bumper state.

gamepad, gamepad_state = connect_gamepad()

# Stick position sliders (display only)
left_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left Stick:',
                                         orientation='vertical', readout=True, readout_format='.2f')
right_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right Stick:',
                                          orientation='vertical', readout=True, readout_format='.2f')

# Bumper state indicators -- lit while held, dark when released
left_bumper_indicator = widgets.Valid(value=False, description='Left Bumper (LB)')
right_bumper_indicator = widgets.Valid(value=False, description='Right Bumper (RB)')

display(widgets.HBox([left_stick_slider, right_stick_slider]))
display(widgets.VBox([left_bumper_indicator, right_bumper_indicator]))

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 3. Start driving -- this also keeps the GUI above updated live.

def update_teleop_gui(left_value, right_value):
    left_stick_slider.value = left_value
    right_stick_slider.value = right_value
    left_bumper_indicator.value = gamepad_state.left_bumper_down
    right_bumper_indicator.value = gamepad_state.right_bumper_down

start_control_loop(rvr, gamepad_state, on_update=update_teleop_gui)

Drive around and test the sticks and bumpers. Confirm: stick sliders track smoothly and return to zero when released, and each bumper indicator lights up exactly while that bumper is held down.

When done, run the cell below to stop.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 4. Stop driving.
stop_control_loop()

## Section 3: Teleoperation

This section tests gamepad connectivity: left/right stick positions, the left/right bumpers, and driving the RVR from the gamepad via a single control-loop thread (see `gamepad_utils.py` for why a dedicated thread is used instead of acting directly on the gamepad library's own background thread).

The GUI below shows live stick position (sliders) and live bumper state (lit while held, dark when released) -- useful for confirming the controller is mapped correctly before relying on it in a student notebook. Bumpers are not wired to any capture/image logic here -- this section is connectivity/state testing only.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 1. Import our shared gamepad + teleop control loop helpers
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 2. Connect the gamepad, and build a GUI showing live stick position and bumper state.

gamepad, gamepad_state = connect_gamepad()

# Stick position sliders (display only)
left_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left Stick:',
                                         orientation='vertical', readout=True, readout_format='.2f')
right_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right Stick:',
                                          orientation='vertical', readout=True, readout_format='.2f')

# Bumper state indicators -- lit while held, dark when released
left_bumper_indicator = widgets.Valid(value=False, description='Left Bumper (LB)')
right_bumper_indicator = widgets.Valid(value=False, description='Right Bumper (RB)')

display(widgets.HBox([left_stick_slider, right_stick_slider]))
display(widgets.VBox([left_bumper_indicator, right_bumper_indicator]))

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 3. Start driving -- this also keeps the GUI above updated live.

def update_teleop_gui(left_value, right_value):
    left_stick_slider.value = left_value
    right_stick_slider.value = right_value
    left_bumper_indicator.value = gamepad_state.left_bumper_down
    right_bumper_indicator.value = gamepad_state.right_bumper_down

start_control_loop(rvr, gamepad_state, on_update=update_teleop_gui)

Drive around and test the sticks and bumpers. Confirm: stick sliders track smoothly and return to zero when released, and each bumper indicator lights up exactly while that bumper is held down.

When done, run the cell below to stop.

In [ ]:
#### ------> RUN THIS CELL WITHOUT CHANGING IT'S CONTENTS <-----#####
# Step 4. Stop driving.
stop_control_loop()